# SQL query from table names - Continued

In [41]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [43]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('Token')

In [22]:
!pip -q install openai

## The old Prompt

In [24]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [25]:
context = [ {'role':'system', 'content':"""
You are an expert Text-to-SQL assistant.

Task:
Convert the user's natural-language question into one valid SQLite query.

Rules:
1. Use only the tables and columns defined in the schema below.
2. Do not invent tables or columns.
3. Prefer simple and efficient SQL.
4. Use explicit JOIN conditions.
5. When the question cannot be answered from the schema, say: "I need a question that can be answered with the provided SQL schema."
6. Return the SQL inside a ```sql code block, followed by a short explanation.

Database schema:

CREATE TABLE employees (
    ID_usr INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    hire_date DATE,
    city TEXT
);

-- sample rows from employees
-- ID_usr | name          | department_id | hire_date  | city
-- 1      | Ana Gomez     | 10            | 2020-02-01 | Madrid
-- 2      | Tom Müller    | 20            | 2019-07-15 | Berlin
-- 3      | Li Wang       | 10            | 2021-03-10 | Munich

CREATE TABLE salary (
    ID_usr INTEGER,
    year INTEGER,
    salary REAL,
    bonus REAL,
    FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);

-- sample rows from salary
-- ID_usr | year | salary | bonus
-- 1      | 2023 | 65000  | 4000
-- 2      | 2023 | 72000  | 5000
-- 3      | 2023 | 69000  | 4500

CREATE TABLE studies (
    ID INTEGER PRIMARY KEY,
    ID_usr INTEGER,
    educational_level TEXT,
    Institution TEXT,
    Years INTEGER,
    Speciality TEXT,
    FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);

-- sample rows from studies
-- ID | ID_usr | educational_level | Institution              | Years | Speciality
-- 1  | 1      | Master            | Technical University     | 2     | Data Science
-- 2  | 2      | Bachelor          | Business School          | 3     | Finance
-- 3  | 3      | PhD               | Technical University     | 4     | Artificial Intelligence

CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT,
    location TEXT
);

-- sample rows from departments
-- department_id | department_name | location
-- 10            | AI Engineering  | Frankfurt
-- 20            | Finance         | Berlin

CREATE TABLE projects (
    project_id INTEGER PRIMARY KEY,
    project_name TEXT,
    department_id INTEGER,
    start_date DATE,
    end_date DATE,
    project_type TEXT,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

-- sample rows from projects
-- project_id | project_name       | department_id | project_type
-- 100        | Customer RAG Bot   | 10            | AI
-- 101        | Payroll Automation | 20            | Automation

CREATE TABLE project_assignments (
    ID_usr INTEGER,
    project_id INTEGER,
    role_on_project TEXT,
    allocation_percentage REAL,
    FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr),
    FOREIGN KEY (project_id) REFERENCES projects(project_id)
);

-- sample rows from project_assignments
-- ID_usr | project_id | role_on_project | allocation_percentage
-- 1      | 100        | ML Engineer     | 80
-- 3      | 100        | AI Architect    | 60
-- 2      | 101        | Analyst         | 50

CREATE TABLE performance_reviews (
    review_id INTEGER PRIMARY KEY,
    ID_usr INTEGER,
    review_year INTEGER,
    rating REAL,
    comments TEXT,
    FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);

-- sample rows from performance_reviews
-- review_id | ID_usr | review_year | rating | comments
-- 1         | 1      | 2023        | 4.5    | strong delivery
-- 2         | 2      | 2023        | 4.0    | reliable analyst
-- 3         | 3      | 2023        | 4.8    | excellent AI expertise

Important relationships:
- employees.ID_usr joins to salary.ID_usr
- employees.ID_usr joins to studies.ID_usr
- employees.ID_usr joins to project_assignments.ID_usr
- employees.ID_usr joins to performance_reviews.ID_usr
- employees.department_id joins to departments.department_id
- departments.department_id joins to projects.department_id
- projects.project_id joins to project_assignments.project_id
"""} ]


In [26]:
# FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
Use the examples below as style guidance.

Example 1
Question: Which employee has the highest salary in 2023?
SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = 2023
ORDER BY s.salary DESC
LIMIT 1;
```
Explanation: This joins employees with salary and sorts salaries from highest to lowest.

Example 2
Question: Which institution has the highest average salary?
SQL:
```sql
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN salary sa ON st.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```
Explanation: This groups employees by institution and calculates the average salary for each institution.

Example 3
Question: List employees working on AI projects with their project names.
SQL:
```sql
SELECT e.name, p.project_name, pa.role_on_project
FROM employees e
JOIN project_assignments pa ON e.ID_usr = pa.ID_usr
JOIN projects p ON pa.project_id = p.project_id
WHERE p.project_type = 'AI';
```
Explanation: This uses project_assignments as the bridge table between employees and projects.

Example 4
Question: What is the average performance rating by department in 2023?
SQL:
```sql
SELECT d.department_name, AVG(pr.rating) AS avg_rating
FROM departments d
JOIN employees e ON d.department_id = e.department_id
JOIN performance_reviews pr ON e.ID_usr = pr.ID_usr
WHERE pr.review_year = 2023
GROUP BY d.department_name;
```
Explanation: This joins departments, employees, and performance reviews to aggregate ratings by department.
"""})


In [34]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [45]:
# New prompt example 1
context_user = context.copy()
print(return_CCRMSQL("""Which employee has the highest salary in 2023?""", context_user))


```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = 2023
ORDER BY s.salary DESC
LIMIT 1;
```
Explanation: This query joins the employees table with the salary table on the employee ID, filters the data for the year 2023, and then retrieves the employee with the highest salary in 2023.


In [46]:
# Old prompt example 1
old_context_user = old_context.copy()
print(return_CCRMSQL("Which employee has the highest salary in 2023?", old_context_user))


This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2023'
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee with the highest salary in the year 2023 by joining the "employees" and "salary" tables on the employee ID. It then filters the results to only include salaries from 2023, orders them in descending order, and limits the output to only the top result.


In [47]:
# New prompt example 2
print(return_CCRMSQL("Which institution has the highest average salary?", context_user))


```sql
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN salary sa ON st.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```
Explanation: This query joins the studies and salary tables to calculate the average salary for each institution, then selects the institution with the highest average salary.


In [48]:
# Old prompt example 2
print(return_CCRMSQL("Which institution has the highest average salary?", old_context_user))


This is your SQL:
```sql
SELECT s.Institution, AVG(s.salary) AS avg_salary
FROM salary s
JOIN employees e ON s.ID_usr = e.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query retrieves the institution with the highest average salary by joining the `salary` table with the `employees` table on the `ID_usr` column. It calculates the average salary for each institution, orders the results in descending order of average salary, and limits the output to the institution with the highest average salary.


# Exercise

Complete the prompts similar to what we did in class. Try at least three versions, compare what works well, and write a one-page report.


## Exercise solution: Prompt versions to test

Below are four prompt experiments. The first three are good prompt versions. The fourth is intentionally weak and is useful for comparing hallucination risk.


In [49]:
# Prompt Version 1: simple but specific question
# Expected behavior: The model should use employees, salary, and departments.
question_v1 = """
List the employee name, department name, and 2023 salary for all employees earning more than 65000.
"""

print(return_CCRMSQL(question_v1, context_user))


```sql
SELECT e.name, d.department_name, s.salary
FROM employees e
JOIN departments d ON e.department_id = d.department_id
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary > 65000 AND s.year = 2023;
```

Explanation: This query retrieves the employee name, department name, and 2023 salary by joining the employees, departments, and salary tables. It filters the results to include only employees earning more than 65000 in 2023.


In [50]:
# Prompt Version 2: aggregation question
# Expected behavior: The model should use employees, salary, and departments.
question_v2 = """
What is the average 2023 salary by department? Show department name and average salary.
"""

print(return_CCRMSQL(question_v2, context_user))


```sql
SELECT d.department_name, AVG(s.salary) AS avg_salary
FROM departments d
JOIN employees e ON d.department_id = e.department_id
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = 2023
GROUP BY d.department_name;
```

Explanation: This query joins departments, employees, and salary tables to calculate the average salary for each department in the year 2023.


In [51]:
# Prompt Version 3: creative multi-table question
# Expected behavior: The model should use employees, salary, studies, project_assignments, and projects.
question_v3 = """
Find employees with a Master or PhD who worked on AI projects and earned more than 65000 in 2023.
Return employee name, educational level, institution, project name, and salary.
"""

print(return_CCRMSQL(question_v3, context_user))


```sql
SELECT e.name, st.educational_level, st.Institution, p.project_name, sa.salary
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN project_assignments pa ON e.ID_usr = pa.ID_usr
JOIN projects p ON pa.project_id = p.project_id
JOIN salary sa ON e.ID_usr = sa.ID_usr
WHERE (st.educational_level = 'Master' OR st.educational_level = 'PhD')
AND p.project_type = 'AI'
AND sa.salary > 65000
AND sa.year = 2023;
```

Explanation: This query retrieves the employee name, educational level, institution, project name, and salary by joining the necessary tables based on the conditions specified in the question.


In [52]:
# Prompt Version 4: weak / hallucination test
# This question cannot be answered because there is no skills table or Python skill column.
# A good model should not invent a table like skills or employee_skills.
question_v4 = """
Which employees are experts in Python?
"""

print(return_CCRMSQL(question_v4, context_user))


```sql
I need a question that can be answered with the provided SQL schema.
```
Explanation: The schema does not contain information about employees' expertise in specific programming languages like Python.


## One-page report: Findings

### Objective

The objective of this lab was to compare prompt designs for generating SQL from natural-language questions. The old prompt provides table descriptions in a loose JSON-like format, while the new prompt uses SQL `CREATE TABLE` definitions, sample rows, explicit relationships, rules, and few-shot examples. The goal was to check whether a better prompt reduces ambiguity and hallucination.

### Prompt versions tested

I tested four versions. Version 1 was a simple filtering question: list employee names, department names, and 2023 salary for employees earning more than 65000. This prompt should use `employees`, `departments`, and `salary`. Version 2 was an aggregation question: average salary by department. This required a `GROUP BY` query and the same three tables. Version 3 was a more creative multi-table question: find employees with a Master or PhD who worked on AI projects and earned more than 65000 in 2023. This required several joins across `employees`, `studies`, `salary`, `project_assignments`, and `projects`. Version 4 was intentionally weak and unanswerable: asking which employees are Python experts, even though the schema has no skills table or skill column.

### What worked well

The new prompt worked better because it described the database using realistic SQL DDL syntax. The `CREATE TABLE` style made column names, data types, primary keys, and foreign-key relationships clearer. The sample rows helped the model understand the meaning of columns such as `project_type`, `educational_level`, and `rating`. The few-shot examples also helped the model follow the desired style: valid SQLite, explicit joins, and a short explanation.

### What did not work well

The old prompt was weaker because the schema description was inconsistent. Some fields used `nombre`, others used `name`; some table names used `tableName`, while another used `tablename`. There was also a missing comma in the `studies` table description. This kind of inconsistency can lead the model to generate incorrect column names such as `ID_Usr` instead of `ID_usr`, or to miss necessary joins. Weak natural-language instructions can also encourage the model to invent tables or columns when the requested information is not in the schema.

### Hallucination observations

The hallucination test was the Python-expert question. Since the schema does not include skills, certifications, resumes, or programming-language columns, the correct behavior is to refuse or ask for a question answerable with the provided schema. If the model generates a query using a fake table such as `skills`, `employee_skills`, or a fake column such as `python_experience`, that is a hallucination.

### What I learned

I learned that text-to-SQL prompting depends heavily on schema clarity. The model performs better when the prompt includes table definitions, column names, relationships, sample rows, constraints, and examples. I also learned that few-shot prompting is useful because examples guide the model toward the expected SQL style. Finally, I learned that prompts should explicitly say not to invent tables or columns, because LLMs may otherwise create plausible but nonexistent schema elements.
